In [25]:
!pip install -q pypdf
!pip install -q pinecone

# Doc_Chat
Document chatting AI Application

In [26]:
# Importing Libraries and tools
from sentence_transformers import SentenceTransformer
from google import genai
from pinecone import Pinecone, ServerlessSpec
from pypdf import PdfReader

from google.colab import userdata

print('Libraries Imported Succesfully')

Libraries Imported Succesfully


In [27]:
model = SentenceTransformer('all-mpnet-base-v2')
print('Sentence Transformer Created!')

gkey = userdata.get('gkey')
llm = genai.Client(api_key=gkey)
print('OpenAI model Created!')

key = userdata.get('pc_key')
pc = Pinecone(api_key=key)
index_name = "my-index"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"Index '{index_name}' created successfully.")
else:
    print(f"Index '{index_name}' already exists.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Sentence Transformer Created!
OpenAI model Created!
Index 'my-index' already exists.


In [28]:
# Defining Functions

# Function that extract content from pdf
def extract_text_from_pdf(pdf_path):
    """
    Reads a PDF file and returns all extracted text as a single string.
    """
    reader = PdfReader(pdf_path)
    full_text = ""

    for page in reader.pages:
        text = page.extract_text()
        if text:
            full_text += text + "\n\n"

    return full_text

# Function that create Chunks from raw text
def chunk_by_paragraphs(text, max_size=1000):
    """Split text by paragraphs, combining small ones"""
    paragraphs = text.split('\n\n')
    chunks = []
    current_chunk = ""

    for para in paragraphs:
        if len(current_chunk) + len(para) < max_size:
            current_chunk += para + "\n\n"
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = para + "\n\n"

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

In [29]:
# Workflow

# 1. Text Extraction from PDF
text_extracted = extract_text_from_pdf('/content/Rich Dad Poor Dad.pdf')
print('text extracted from pdf')

# 2. Creating Chunks from the raw text
chunks = chunk_by_paragraphs(text_extracted)
print('chunks created')

# 3. Creating Embeddings from Chunks
embeddings = model.encode(chunks).tolist()
print("Embeddings Generated")

# 4. Connecting the PineCone Index and upserting the embedded vectors
index = pc.Index(index_name)

vectors = []
n = len(chunks)
for i in range(1,n+1):
    vectors.append({
        "id": f"chunk-{i}",
        "values": embeddings[i-1],
        "metadata": {"text": chunks[i-1]}
    })

# Upserting vectors as batch of 100
batch_size = 100
for i in range(0, n, batch_size):
    batch = vectors[i:i + batch_size]
    index.upsert(vectors=batch)

print(f"Successfully uploaded {len(chunks)} chunks to Pinecone index '{index_name}'!")

text extracted from pdf
chunks created
Embeddings Generated
Successfully uploaded 234 chunks to Pinecone index 'my-index'!


In [30]:
# Getting query and transform
query = input("Enter Your Query Here: ")
# 1. Ensure embedding is a list and query Pinecone
query_embedded = model.encode(query).tolist()
results = index.query(vector=query_embedded, top_k=3, include_metadata=True)

# 2. Extract the text chunks safely from the matches
retrieved_texts = []
for match in results['matches']:
    text = match['metadata']['text']
    retrieved_texts.append(text)

# 3. Pass them dynamically into your prompt
prompt = f"""
Answer the question based on the context below.

Rules:
- Use only information from the context
- Cite the document number you're referencing
- If uncertain, say
"I don't know based on the provided context"
- Keep answers concise and factual

Context:
[1] {retrieved_texts[0]}
[2] {retrieved_texts[1]}
[3] {retrieved_texts[2]}

Question: {query}
"""

Enter Your Query Here: How to increase financial IQ


In [33]:
def generate_response(prompt: str) -> str:
    response = llm.models.generate_content(
        model="gemini-3.8-flash",  # Fast and free-tier friendly model
        contents=prompt
    )
    return response.text

# Test your RAG answer generation
answer = generate_response(prompt)
print(answer)

Based on the provided context, you can increase your financial IQ/intelligence by:

* Combining and synergizing "four technical skills" (or becoming familiar with the "four pillars of financial intelligence") [1], [3].
* Learning about the benefits and protection that legal entities and corporations offer for businesses and assets [1].
* Developing it over a lifetime by continually learning, welcoming change, and gaining experience and wisdom over the years [2], [3].
